<a href="https://colab.research.google.com/github/raj-aryan7/ProteinBERT/blob/main/notebooks/LSTM_without_pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM without Pretraining

Dataset: TAPE Secondary Structure  
Model: Bidirectional LSTM  
Epochs: 5  
Validation Accuracy: ~0.71  

This notebook reproduces the baseline LSTM experiment used in the ProteinBERT paper.

In [3]:
!pip install tape-proteins
!pip install biopython
!pip install torch torchvision
!pip install pandas numpy scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.4 MB/s eta 0:00:00


In [4]:
!pip install tape-proteins
!pip install lmdb

In [5]:
!wget https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
!tar -xzf secondary_structure.tar.gz
!mkdir data
!mv secondary_structure data/

--2026-03-09 18:08:57--  https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.199.104, 3.5.22.84, 54.231.161.200, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.199.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 251794897 (240M) [application/x-tar]
Saving to: ‘secondary_structure.tar.gz’

secondary_structure 100%[===================>] 240.13M  12.8MB/s    in 21s     

2026-03-09 18:09:19 (11.6 MB/s) - ‘secondary_structure.tar.gz’ saved [251794897/251794897]



In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

from tape.datasets import SecondaryStructureDataset
from tape import TAPETokenizer

from tqdm import tqdm

In [9]:
!ls data/secondary_structure

secondary_structure_casp12.lmdb  secondary_structure_ts115.lmdb
secondary_structure_cb513.lmdb	 secondary_structure_valid.lmdb
secondary_structure_train.lmdb


In [12]:
train_dataset = SecondaryStructureDataset(
    data_path='./data',
    split='train'
)

valid_dataset = SecondaryStructureDataset(
    data_path='./data',
    split='valid'
)

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))

Train size: 8678
Valid size: 2170


In [13]:
tokenizer = TAPETokenizer(vocab='iupac')

In [27]:
def collate_fn(batch):

    input_ids = [torch.tensor(item[0]) for item in batch]
    labels = [torch.tensor(item[2]) for item in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-1)

    return input_ids, labels

In [28]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

In [29]:
class LSTMModel(nn.Module):

    def __init__(
        self,
        vocab_size=30,
        embed_dim=128,
        hidden_dim=256,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        x, _ = self.lstm(x)

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel().to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [31]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in tqdm(train_loader):

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs.view(-1, 3),
            labels.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

100%|██████████| 543/543 [00:23<00:00, 23.19it/s]


Epoch 1 Loss: 0.7790


100%|██████████| 543/543 [00:23<00:00, 23.27it/s]


Epoch 2 Loss: 0.7142


100%|██████████| 543/543 [00:23<00:00, 23.24it/s]


Epoch 3 Loss: 0.6940


100%|██████████| 543/543 [00:22<00:00, 23.62it/s]


Epoch 4 Loss: 0.6799


100%|██████████| 543/543 [00:23<00:00, 23.16it/s]

Epoch 5 Loss: 0.6704


In [32]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in valid_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        preds = outputs.argmax(dim=-1)

        mask = labels != -1

        correct += (preds[mask] == labels[mask]).sum().item()
        total += mask.sum().item()

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.7106697238975744


In [33]:
torch.save(model.state_dict(), "lstm_without_pretraining.pt")

In [34]:
print("Experiment: LSTM without pretraining")
print("Dataset: TAPE Secondary Structure")
print("Train samples:", len(train_dataset))
print("Validation samples:", len(valid_dataset))
print("Epochs:", epochs)
print("Final Validation Accuracy:", accuracy)

Experiment: LSTM without pretraining
Dataset: TAPE Secondary Structure
Train samples: 8678
Validation samples: 2170
Epochs: 5
Final Validation Accuracy: 0.7106697238975744


In [35]:
model.eval()

val_loss = 0

with torch.no_grad():

    for inputs, labels in valid_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        loss = criterion(
            outputs.view(-1, 3),
            labels.view(-1)
        )

        val_loss += loss.item()

print("Validation Loss:", val_loss/len(valid_loader))

Validation Loss: 0.6727846624220118
